## 1. Imports and paths

In [124]:
from pathlib import Path
import numpy as np
import pandas as pd

from pyBKT.models import Model

BASE_DIR = Path("..").resolve()          # app/
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"
MODELS_DIR = BASE_DIR / "models"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

BASE_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app
PROCESSED_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/processed
OUTPUTS_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs


## 2. Load cleaned dataset

In [125]:
TRAIN_06_PATH = PROCESSED_DIR / "df_train_kc_cleaned_2006-2007_v.1.csv"
TRAIN_08_PATH = PROCESSED_DIR / "df_train_kc_cleaned_2008-2009_v.1.csv"

df_06 = pd.read_csv(TRAIN_06_PATH, encoding="latin")
df_08 = pd.read_csv(TRAIN_08_PATH, encoding="latin")

print("2006-07 shape:", df_06.shape)
print("2008-09 shape:", df_08.shape)

print(df_06.columns.tolist())

2006-07 shape: (205323, 11)
2008-09 shape: (542777, 11)
['Row', 'Anon Student Id', 'Problem Name', 'Step Name', 'Step Start Time', 'Step End Time', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'skill_name']


## 3. Helpers and mapping

In [126]:
KC_NAME_MAP = {
    "Expand / Eliminate Parentheses": "expand_eliminate_parentheses",
    "Combine Like Terms": "combine_like_terms",
    "Move Constants Across the Equation": "move_constants",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

COLUMNS_MAP = {
    "Anon Student Id": "user_id",
    "skill_name": "skill_name",
    "Correct First Attempt": "correct",
    "Problem Name": "problem_id",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

BASE_DEFAULTS = {
    "order_id": "order_id",
    "user_id": "user_id",
    "skill_name": "skill_name",
    "correct": "correct",
}

In [127]:
def prepare_columns(df):
    df = df.copy()
    time_cols = [
        "Step Start Time",
    ]
    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


## 4. Build the dataframe for pyBKT

This function transforms the cleaned datasets into the format expected by pyBKT.

In [128]:
def build_pybkt_df(df,dataset_tag,min_seq_len=2):
    data = df.copy()

    required_cols = [
        "Anon Student Id",
        "skill_name",
        "Correct First Attempt",
        "Step Start Time"
    ]
    missing = [c for c in required_cols if c not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = prepare_columns(data)

    # Keep only valid rows
    data = data.dropna(subset=[
        "Anon Student Id",
        "skill_name",
        "Correct First Attempt",
        "Step Start Time"
    ]).copy()

    data["Correct First Attempt"] = pd.to_numeric(data["Correct First Attempt"], errors="coerce"
    )
    data = data[data["Correct First Attempt"].isin([0, 1])].copy()

    # Keep only the final selected KCs
    data = data[data["skill_name"].isin(KC_NAME_MAP.keys())].copy()

    # Stable IDs
    data["user_id"] = dataset_tag + "__" + data["Anon Student Id"].astype(str)
    data["skill_name"] = data["skill_name"].map(KC_NAME_MAP)
    data["correct"] = data["Correct First Attempt"].astype(int)

    sort_cols = ["user_id", "skill_name", "Step Start Time"]
    if "Row" in data.columns:
        sort_cols.append("Row")

    data = data.sort_values(sort_cols).reset_index(drop=True)

    # Sequential order_id within each student-skill sequence
    data["order_id"] = data.groupby(["user_id", "skill_name"]).cumcount() + 1

    # Remove sequences that are too short
    seq_sizes = (
        data.groupby(["user_id", "skill_name"])
        .size()
        .reset_index(name="n_obs")
    )
    keep_seq = seq_sizes[seq_sizes["n_obs"] >= min_seq_len][["user_id", "skill_name"]]

    data = data.merge(keep_seq, on=["user_id", "skill_name"], how="inner")

    keep_cols = ["order_id","user_id","skill_name","correct"]

    return data[keep_cols].reset_index(drop=True)

## 5. Create the three modelling datasets

The notebook evaluates:

- `2006-07`
- `2008-09`
- `2006-09`, created by concatenating both datasets

In [129]:
bkt_06 = build_pybkt_df(df_06, dataset_tag="2006_07")
bkt_08 = build_pybkt_df(df_08, dataset_tag="2008_09")
bkt_06_09 = pd.concat([bkt_06, bkt_08], ignore_index=True)

datasets_bkt = {
    "2006-07": bkt_06,
    "2008-09": bkt_08,
    "2006-09": bkt_06_09,
}

for name, df_ in datasets_bkt.items():
    print(
        f"{name}: rows={len(df_)}, students={df_['user_id'].nunique()}, "
        f"skills={df_['skill_name'].nunique()}"
    )

2006-07: rows=205194, students=945, skills=5
2008-09: rows=542560, students=2113, skills=5
2006-09: rows=747754, students=3058, skills=5


 ## 6. Define models 
 Only these two models are tested:

- `basic`
- `forgets`

Both return 5 probabilities per KC:

- `prior`
- `learns`
- `guesses`
- `slips`
- `forgets`

In the `basic` model, `forgets` is fixed at 0.

In [130]:
MODEL_SPECS = {
    "basic": {
        "defaults_extra": {},
        "fit_kwargs": {}
    },
    "forgets": {
        "defaults_extra": {},
        "fit_kwargs": {"forgets": True}
    },
}

## 7. Evaluation functions

In [131]:
def summarize_cv_output(cv_output):
    """
    pyBKT crossvalidate can return a scalar, list, array, Series or DataFrame.
    This function summarises the output using mean and standard deviation.
    """
    if isinstance(cv_output, pd.DataFrame):
        numeric = cv_output.select_dtypes(include=[np.number])
        arr = numeric.to_numpy().ravel()
    elif isinstance(cv_output, (list, tuple, np.ndarray, pd.Series)):
        arr = np.array(cv_output, dtype=float).ravel()
    else:
        arr = np.array([cv_output], dtype=float)

    arr = arr[~np.isnan(arr)]

    if len(arr) == 0:
        return np.nan, np.nan

    return float(np.mean(arr)), float(np.std(arr))


def run_one_model_on_one_dataset(
    df,
    dataset_name,
    model_name,
    model_spec,
    folds=3,
    seed=42,
    num_fits=1
):
    defaults = BASE_DEFAULTS.copy()
    defaults.update(model_spec["defaults_extra"])

    fit_kwargs = model_spec["fit_kwargs"].copy()

    row = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "n_rows": len(df),
        "n_students": df["user_id"].nunique(),
        "n_skills": df["skill_name"].nunique(),
    }

    # Fit model
    model = Model(seed=seed, num_fits=num_fits)
    model.fit(data=df, defaults=defaults, **fit_kwargs)

    # Extract learned parameters
    params = model.params().copy().reset_index()


    # Robust renaming
    rename_map = {}
    if "skill" in params.columns:
        rename_map["skill"] = "skill_name"
    if "param" in params.columns:
        rename_map["param"] = "parameter"
    if "value" in params.columns:
        rename_map["value"] = "probability"

    params = params.rename(columns=rename_map)

    # If pyBKT returns unnamed index levels, handle them
    unnamed_cols = [c for c in params.columns if str(c).startswith("level_")]
    if "skill_name" not in params.columns and len(unnamed_cols) >= 1:
        params = params.rename(columns={unnamed_cols[0]: "skill_name"})
    if "parameter" not in params.columns and len(unnamed_cols) >= 2:
        params = params.rename(columns={unnamed_cols[1]: "parameter"})
    if "class" not in params.columns and len(unnamed_cols) >= 3:
        params = params.rename(columns={unnamed_cols[2]: "class"})

    params["dataset_name"] = dataset_name
    params["model_name"] = model_name

    # Predictions and train-set evaluation
    train_preds = model.predict(data=df)
    train_rmse = model.evaluate(data=df, metric="rmse")
    train_auc = model.evaluate(data=df, metric="auc")
    train_acc = model.evaluate(data=df, metric="accuracy")

    # Cross-validation: AUC
    cv_model_auc = Model(seed=seed, num_fits=num_fits)
    cv_auc_raw = cv_model_auc.crossvalidate(
        data=df,
        defaults=defaults,
        folds=folds,
        metric="auc",
        **fit_kwargs
    )
    cv_auc_mean, cv_auc_std = summarize_cv_output(cv_auc_raw)

    # Cross-validation: RMSE
    cv_model_rmse = Model(seed=seed, num_fits=num_fits)
    cv_rmse_raw = cv_model_rmse.crossvalidate(
        data=df,
        defaults=defaults,
        folds=folds,
        metric="rmse",
        **fit_kwargs
    )
    cv_rmse_mean, cv_rmse_std = summarize_cv_output(cv_rmse_raw)

    # Cross-validation: Accuracy
    cv_model_acc = Model(seed=seed, num_fits=num_fits)
    cv_acc_raw = cv_model_acc.crossvalidate(
        data=df,
        defaults=defaults,
        folds=folds,
        metric="accuracy",
        **fit_kwargs
    )
    cv_acc_mean, cv_acc_std = summarize_cv_output(cv_acc_raw)

    row.update({
        "status": "ok",
        "train_auc": float(train_auc),
        "train_rmse": float(train_rmse),
        "train_accuracy": float(train_acc),
        "cv_auc_mean": cv_auc_mean,
        "cv_auc_std": cv_auc_std,
        "cv_rmse_mean": cv_rmse_mean,
        "cv_rmse_std": cv_rmse_std,
        "cv_accuracy_mean": cv_acc_mean,
        "cv_accuracy_std": cv_acc_std,
        "n_train_predictions": len(train_preds),
    })

    return row, params_df, train_preds



## 9. Run the comparison

In [132]:
def run_model_selection(
    datasets_dict,
    model_specs,
    folds=3,
    seed=42,
    num_fits=1
):
    results = []
    params_tables = []
    preds_tables = {}

    for dataset_name, df in datasets_dict.items():
        print("DATASET:", dataset_name)

        for model_name, model_spec in model_specs.items():
            print(f"Running {model_name}...")

            res_row, params_df, train_preds = run_one_model_on_one_dataset(
                df=df,
                dataset_name=dataset_name,
                model_name=model_name,
                model_spec=model_spec,
                folds=folds,
                seed=seed,
                num_fits=num_fits
            )

            results.append(res_row)

            if params_df is not None:
                params_tables.append(params_df)

            if train_preds is not None:
                preds_tables[(dataset_name, model_name)] = train_preds

    results_df = pd.DataFrame(results)

    if params_tables:
        params_df = pd.concat(params_tables, ignore_index=True)
    else:
        params_df = pd.DataFrame()

    return results_df, params_df, preds_tables

In [133]:
results_df, params_df, preds_tables = run_model_selection(
    datasets_dict=datasets_bkt,
    model_specs=MODEL_SPECS,
    folds=3,
    seed=42,
    num_fits=1
)

results_df

DATASET: 2006-07
Running basic...


Running forgets...
DATASET: 2008-09
Running basic...
Running forgets...
DATASET: 2006-09
Running basic...
Running forgets...


,dataset_name,model_name,n_rows,n_students,n_skills,status,train_auc,train_rmse,train_accuracy,cv_auc_mean,cv_auc_std,cv_rmse_mean,cv_rmse_std,cv_accuracy_mean,cv_accuracy_std,n_train_predictions
0,2006-07,basic,205194,945,5,ok,0.70388,0.35702,0.83396,0.67840,0.03606,0.36120,0.04305,0.82733,0.05131,205194
1,2006-07,forgets,205194,945,5,ok,0.72851,0.35300,0.83583,0.70155,0.04855,0.35910,0.04361,0.82792,0.05099,205194
2,2008-09,basic,542560,2113,5,ok,0.70080,0.31951,0.87223,0.67730,0.02348,0.32824,0.05536,0.85471,0.06887,542560
3,2008-09,forgets,542560,2113,5,ok,0.73365,0.31602,0.87223,0.69339,0.03994,0.32570,0.05520,0.85471,0.06887,542560
4,2006-09,basic,747754,3058,5,ok,0.70281,0.33094,0.85974,0.68350,0.01910,0.33791,0.05142,0.84528,0.06476,747754
5,2006-09,forgets,747754,3058,5,ok,0.73410,0.32715,0.85974,0.69342,0.02862,0.33614,0.05097,0.84528,0.06476,747754


## 10. Select the best model

Selection criteria:

1. Highest `cv_auc_mean`
2. Lowest `cv_rmse_mean`
3. If tied, choose the simpler model

In [134]:
MODEL_COMPLEXITY = {
    "basic": 0,
    "forgets": 1,
}

ranked_results = results_df.copy()
ranked_results = ranked_results[ranked_results["status"] == "ok"].copy()

ranked_results["model_complexity"] = ranked_results["model_name"].map(MODEL_COMPLEXITY)

ranked_results = ranked_results.sort_values(
    by=["dataset_name", "cv_auc_mean", "cv_rmse_mean", "model_complexity"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

best_models = (
    ranked_results.groupby("dataset_name", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("BEST MODEL PER DATASET")
display(best_models[[
    "dataset_name",
    "model_name",
    "train_auc",
    "train_rmse",
    "train_accuracy",
    "cv_auc_mean",
    "cv_rmse_mean",
    "cv_accuracy_mean"
]])

print("FULL RANKING")
display(ranked_results[[
    "dataset_name",
    "model_name",
    "train_auc",
    "train_rmse",
    "train_accuracy",
    "cv_auc_mean",
    "cv_rmse_mean",
    "cv_accuracy_mean",
    "model_complexity"
]])

BEST MODEL PER DATASET


,dataset_name,model_name,train_auc,train_rmse,train_accuracy,cv_auc_mean,cv_rmse_mean,cv_accuracy_mean
0,2006-07,forgets,0.72851,0.35300,0.83583,0.70155,0.35910,0.82792
1,2006-09,forgets,0.73410,0.32715,0.85974,0.69342,0.33614,0.84528
2,2008-09,forgets,0.73365,0.31602,0.87223,0.69339,0.32570,0.85471


FULL RANKING


,dataset_name,model_name,train_auc,train_rmse,train_accuracy,cv_auc_mean,cv_rmse_mean,cv_accuracy_mean,model_complexity
0,2006-07,forgets,0.72851,0.35300,0.83583,0.70155,0.35910,0.82792,1
1,2006-07,basic,0.70388,0.35702,0.83396,0.67840,0.36120,0.82733,0
2,2006-09,forgets,0.73410,0.32715,0.85974,0.69342,0.33614,0.84528,1
3,2006-09,basic,0.70281,0.33094,0.85974,0.68350,0.33791,0.84528,0
4,2008-09,forgets,0.73365,0.31602,0.87223,0.69339,0.32570,0.85471,1
5,2008-09,basic,0.70080,0.31951,0.87223,0.67730,0.32824,0.85471,0


- dataset_name: the dataset used to train and evaluate the model. In your table, each row is a different dataset version.
- model_name: the BKT variant selected as best for that dataset. Here, all three best models are forgets, meaning the version that includes a forgetting parameter.
train_auc: AUC on the training data. It tells you how well the model separates correct vs incorrect responses on data it has already seen.
train_rmse: RMSE on the training data. It measures how close the predicted probabilities are to the observed outcomes. Lower is better.
train_accuracy: accuracy on the training data. It is the percentage of responses correctly classified as right or wrong.
cv_auc_mean: mean AUC across cross-validation folds. This is usually the most important column for model selection because it reflects performance on unseen data.
cv_rmse_mean: mean RMSE across cross-validation folds. Lower is better, and it tells you how good the probability estimates are on unseen data.
cv_accuracy_mean: mean accuracy across cross-validation folds. It shows classification performance on unseen data, but it can be less informative if the data are imbalanced.

## 11. Select the overall best dataset–model combination

After selecting the best model **within each dataset**, choose the **overall best candidate** across datasets using the same criteria:

1. Highest `cv_auc_mean`
2. Lowest `cv_rmse_mean`
3. If tied, choose the simpler model

In [135]:
overall_best = best_models.iloc[2]

best_dataset_name = overall_best["dataset_name"]
best_model_name = overall_best["model_name"]

print("OVERALL BEST MODEL")
display(overall_best[[
    "dataset_name",
    "model_name",
    "train_auc",
    "train_rmse",
    "train_accuracy",
    "cv_auc_mean",
    "cv_rmse_mean",
    "cv_accuracy_mean",
    "model_complexity"
]])


OVERALL BEST MODEL


dataset_name        2008-09
model_name          forgets
train_auc           0.73365
train_rmse          0.31602
train_accuracy      0.87223
cv_auc_mean         0.69339
cv_rmse_mean        0.32570
cv_accuracy_mean    0.85471
model_complexity          1
Name: 2, dtype: object

## 12. Display the KC probability table for the selected model

The following table shows the learned BKT probabilities (`prior`, `learns`, `guesses`, `slips`, and `forgets`) for each KC in the selected best model.


In [145]:
# ------------------------------------------------------------
# Rerun "forgets" model on the 2008-09 dataset
# ------------------------------------------------------------
chosen_dataset = "2008-09"
chosen_model_name = "forgets"

df_2008_09 = bkt_08.copy()
model_spec = MODEL_SPECS[chosen_model_name]

defaults = BASE_DEFAULTS.copy()
defaults.update(model_spec["defaults_extra"])

fit_kwargs = model_spec["fit_kwargs"].copy()

model_2008_09_forgets = Model(seed=42, num_fits=1)
model_2008_09_forgets.fit(
    data=df_2008_09,
    defaults=defaults,
    **fit_kwargs
)

# ------------------------------------------------------------
# Extract parameters
# ------------------------------------------------------------
params_2008_09 = model_2008_09_forgets.params().copy().reset_index()

params_2008_09 = params_2008_09.rename(columns={
    "skill": "skill_name",
    "param": "parameter",
    "value": "probability"
})

params_2008_09["dataset_name"] = chosen_dataset
params_2008_09["model_name"] = chosen_model_name


# ------------------------------------------------------------
# Wide format
# ------------------------------------------------------------
params_2008_09_wide = (
    params_2008_09.pivot_table(
        index="skill_name",
        columns="parameter",
        values="probability",
        aggfunc="first"
    )
    .reset_index()
)

print("KC probabilities (wide format)")
display(params_2008_09_wide)

KC probabilities (wide format)


parameter,skill_name,forgets,guesses,learns,prior,slips
0,combine_like_terms,0.02448,0.66119,0.09808,0.68812,0.03112
1,expand_eliminate_parentheses,0.06855,0.57901,0.42183,0.10577,0.00697
2,move_constants,0.01114,0.54285,0.06593,0.35015,0.05654
3,normalize_negative_sign,0.12883,0.31042,0.35410,0.29895,0.04824
4,remove_coefficient,0.01498,0.59559,0.08226,0.39922,0.04837


In [146]:
# ------------------------------------------------------------
# Rerun "forgets" model on the 2008-09 dataset
# ------------------------------------------------------------
chosen_dataset = "2008-09"
chosen_model_name = "forgets"

df_2006_07 = bkt_06.copy()
model_spec = MODEL_SPECS[chosen_model_name]

defaults = BASE_DEFAULTS.copy()
defaults.update(model_spec["defaults_extra"])

fit_kwargs = model_spec["fit_kwargs"].copy()

model_2006_07_forgets = Model(seed=42, num_fits=1)
model_2006_07_forgets.fit(
    data=df_2006_07,
    defaults=defaults,
    **fit_kwargs
)

# ------------------------------------------------------------
# Extract parameters
# ------------------------------------------------------------
params_2006_07 = model_2006_07_forgets.params().copy().reset_index()

params_2006_07 = params_2006_07.rename(columns={
    "skill": "skill_name",
    "param": "parameter",
    "value": "probability"
})

params_2006_07["dataset_name"] = chosen_dataset
params_2006_07["model_name"] = chosen_model_name

# ------------------------------------------------------------
# Wide format
# ------------------------------------------------------------
params_2006_07_wide = (
    params_2006_07.pivot_table(
        index="skill_name",
        columns="parameter",
        values="probability",
        aggfunc="first"
    )
    .reset_index()
)

print("KC probabilities (wide format)")
display(params_2006_07_wide)

KC probabilities (wide format)


parameter,skill_name,forgets,guesses,learns,prior,slips
0,combine_like_terms,0.06551,0.49781,0.31178,0.46626,0.02869
1,expand_eliminate_parentheses,0.05744,0.48175,0.36424,0.09630,0.02124
2,move_constants,0.01487,0.36890,0.08303,0.23268,0.09899
3,normalize_negative_sign,0.20240,0.30973,0.49587,0.57459,0.02624
4,remove_coefficient,0.00954,0.49990,0.07016,0.37736,0.08370


A comparison of the BKT+forget parameter estimates across datasets showed that the 2006–07 dataset generally produced more interpretable models, particularly due to lower guess probabilities in most Knowledge Components. In contrast, the 2008–09 dataset yielded slightly lower slip values and a more plausible parameterization for Normalize Negative Variable / Sign, but still showed relatively high guess probabilities for several other KCs. Overall, the 2006–07 dataset appeared to provide a better balance between plausibility and stability of the estimated parameters.